# Module 06: Generative AI, Large Language Models (LLMs) & RAG Pipelines
## Master Guide: Prompt Engineering, Semantic Search, Vector Stores, and Retrieval-Augmented Generation

Welcome to the cutting-edge capstone of modern NLP! Over the last few years, the frontier of natural language processing has expanded from classification and NER to **Generative AI**, **Large Language Models (LLMs)**, and **Retrieval-Augmented Generation (RAG)**.

While LLMs possess vast parametric knowledge stored in their billions of weights, they have two fundamental limitations:
1. **Knowledge Cutoff & Static Knowledge**: They do not know private internal company documents or real-time news.
2. **Hallucination**: When unsure, LLMs generate plausible-sounding falsehoods with high confidence.

**RAG (Retrieval-Augmented Generation)** is the premier architecture that solves both problems by pairing an external knowledge base (vector database) with a language model!

---

### 🎯 Key Learning Objectives in this Notebook:
1. **LLM Mechanics & Generation Decoding Strategies**
   - Autoregressive text generation: Greedy, Beam Search, Top-$k$, Top-$p$ (Nucleus), Temperature
2. **Prompt Engineering as an Engineering Discipline**
   - Zero-Shot vs Few-Shot In-Context Learning
   - Chain-of-Thought (CoT) Prompting
   - The ReAct Framework (Reasoning + Acting)
3. **Dense Vector Embeddings & Semantic Search**
   - Why TF-IDF and keyword matching fail on semantic queries
   - Cosine Similarity & Vector Space Geometry
4. **Building a Complete End-to-End RAG Pipeline from Scratch in Pure Python**:
   - Document Ingestion & Chunking with sliding windows
   - Building an in-memory Vector Store with dense vector indexing
   - Top-$k$ Nearest Neighbor semantic search
   - Context injection & prompt synthesis
   - Grounded generation that eliminates hallucinations!
5. **Modern Fine-Tuning Overview: LoRA (Low-Rank Adaptation)**
   - Parameter-Efficient Fine-Tuning (PEFT) mathematical principles
6. **Self-Assessment Practice Exercises & Detailed Solutions**


---
## 1. LLM Generation & Decoding Strategies Demystified

Autoregressive models (like GPT-4, Llama, Mistral) generate text token-by-token:
$$P(w_1, w_2, \dots, w_T) = \prod_{t=1}^T P(w_t \mid w_1, w_2, \dots, w_{t-1})$$

At each step $t$, the model outputs raw unnormalized scores (logits) $z_t \in \mathbb{R}^{V}$ for every word in the vocabulary $V$. How we convert these logits into the next selected word is determined by the **Decoding Strategy**:

```text
Logits (z) ---> [ Temperature Scaling: z / T ] ---> [ Top-K Filter ] ---> [ Top-p Filter ] ---> [ Softmax ] ---> Sample next token
```

### Decoding Strategies:
1. **Greedy Search**: Always pick the token with highest probability: $w_t = \arg\max P(w_t)$.
   - *Pros*: Fast, deterministic.
   - *Cons*: Repetitive loops, lacks creativity.
2. **Temperature ($T$)**: Scales logits before softmax: $P(w_i) = \frac{e^{z_i / T}}{\sum e^{z_j / T}}$.
   - $T \to 0$: Approaches greedy search (deterministic, focused).
   - $T = 1.0$: Standard probability distribution.
   - $T > 1.0$: Flattens distribution, increasing randomness and diversity.
3. **Top-$k$ Sampling**: Only sample from the $k$ most probable tokens. Discards low-probability long-tail words.
4. **Top-$p$ (Nucleus) Sampling**: Sets a cumulative probability threshold $p$ (e.g. $p=0.90$). Selects the smallest set of tokens whose cumulative sum exceeds $p$. Dynamically expands and contracts the candidate pool based on model confidence!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax_with_temperature(logits, temperature=1.0):
    scaled = np.array(logits) / max(temperature, 1e-5)
    exp_scaled = np.exp(scaled - np.max(scaled))
    return exp_scaled / np.sum(exp_scaled)

sample_vocab = ["learning", "coding", "coffee", "astronomy", "pineapple"]
sample_logits = [4.5, 3.8, 1.2, 0.5, -1.0]

temps = [0.2, 0.7, 1.5]
plt.figure(figsize=(11, 4))

for idx, t in enumerate(temps):
    probs = softmax_with_temperature(sample_logits, temperature=t)
    plt.subplot(1, 3, idx + 1)
    plt.bar(sample_vocab, probs, color='royalblue', alpha=0.8)
    plt.title(f"Temperature T = {t}")
    plt.ylabel("Probability")
    plt.ylim(0, 1.0)
    plt.xticks(rotation=30)
    plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


---
## 2. Prompt Engineering Patterns

Prompt engineering is the deliberate design of input prompts to guide an LLM's reasoning and output format.

### 1. Few-Shot In-Context Learning:
Giving the model 2-3 exemplar input-output pairs inside the prompt teaches it the desired style and logic without changing any model weights.

### 2. Chain-of-Thought (CoT) Prompting:
Encouraging the model to generate intermediate reasoning steps before providing the final answer ("*Think step by step*").
- Standard Prompt: *"Q: Roger has 5 tennis balls. He buys 2 cans of tennis balls. Each can has 3 balls. How many does he have? A: 11"*
- The explicit reasoning chain drastically reduces arithmetic and logic errors!

### 3. Structured Output (JSON Prompting):
Ensuring production systems receive machine-parseable data:
```text
System: You are an expert code reviewer. Extract issues from the provided code.
Output your response STRICTLY as a JSON object adhering to this schema:
{
  "has_bugs": boolean,
  "severity": "low" | "medium" | "high",
  "explanation": string,
  "suggested_fix": string
}
```


---
## 3. Semantic Search & Vector Embeddings

Classical keyword search (BM25, TF-IDF) checks for exact word matches. If a user searches for:
- Query: *"How to cure a headache?"*
- Document: *"Remedies for migraine relief"*
Keyword search finds **zero overlap**!

**Dense Vector Embeddings** map sentences into a continuous high-dimensional geometric space where sentences with similar meanings have vectors close to each other, regardless of differing vocabulary!

### Cosine Similarity:
$$\text{CosineSimilarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \frac{\sum_{i=1}^d u_i v_i}{\sqrt{\sum_{i=1}^d u_i^2} \sqrt{\sum_{i=1}^d v_i^2}}$$

- Value is in $[-1.0, 1.0]$.
- \\$1.0$: Identical direction (identical semantic meaning).
- \\\$0.0$: Orthogonal (unrelated).
- $-1.0$: Opposite direction.


In [ ]:
def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))

# Let's test cosine similarity on sample semantic embeddings
vec_cat = np.array([0.9, 0.8, 0.1, 0.0])
vec_kitten = np.array([0.85, 0.82, 0.15, 0.05])
vec_rocket = np.array([0.05, 0.1, 0.95, 0.88])

print(f"Similarity('cat', 'kitten'): {cosine_similarity(vec_cat, vec_kitten):.4f} (Very High)")
print(f"Similarity('cat', 'rocket'): {cosine_similarity(vec_cat, vec_rocket):.4f} (Very Low)")


---
## 4. Building an End-to-End RAG Pipeline from Scratch in Pure Python

Let's build a complete, transparent, zero-dependency RAG system!

### Architecture of Our RAG System:
```text
[Knowledge Documents]
        |
    [Chunking]  --> Divide documents into clean, semantic passages
        |
  [Embedding]   --> Transform text chunks into dense vectors
        |
  [Vector DB]   --> In-memory vector store indexing chunks + vectors
        |
[User Query] -> [Query Embedding] -> [Top-K Vector Search] -> [Retrieved Chunks]
                                                                     |
                                                           [Context Injection]
                                                                     |
                                                             [LLM Answer Output]
```


In [ ]:
import re
from typing import List, Dict, Any

# 1. Knowledge Base (Internal documentation of a deep learning research project)
knowledge_documents = [
    {
        "doc_id": "doc_01",
        "title": "Optimizer Guidelines",
        "content": "For training deep transformers, AdamW with decoupled weight decay is strictly recommended. We typically set the initial learning rate between 1e-5 and 5e-5 with a linear warmup over the first 10% of total training steps. Gradient clipping with max_norm=1.0 is essential to prevent exploding gradients."
    },
    {
        "doc_id": "doc_02",
        "title": "Attention and Softmax Scaling",
        "content": "Scaled dot-product attention divides Q @ K.T by sqrt(d_k). This scaling factor prevents the magnitude of dot products from growing excessively large in high dimensions. Without it, softmax saturates into regions with near-zero gradients, completely stalling backpropagation."
    },
    {
        "doc_id": "doc_03",
        "title": "Model Serving and Latency Optimization",
        "content": "For low-latency production deployment of Transformer models, we apply FP16 mixed precision quantization and ONNX Runtime conversion. Additionally, Key-Value (KV) caching in decoder generation avoids redundant recalculation of attention states for past tokens."
    },
    {
        "doc_id": "doc_04",
        "title": "Handling Imbalanced Datasets",
        "content": "When dealing with extreme class imbalance in text classification, standard accuracy is misleading. Engineers should evaluate using Precision-Recall AUC (PR-AUC) and macro-averaged F1-Score, or apply Focal Loss and class-weighted CrossEntropy."
    }
]

# 2. Document Chunking
def chunk_document(text: str, max_words: int = 40, overlap: int = 10) -> List[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += max_words - overlap
        if end >= len(words):
            break
    return chunks

all_chunks = []
for doc in knowledge_documents:
    doc_chunks = chunk_document(doc["content"])
    for i, c in enumerate(doc_chunks):
        all_chunks.append({
            "chunk_id": f"{doc['doc_id']}_c{i}",
            "doc_title": doc["title"],
            "text": c
        })

print(f"Generated {len(all_chunks)} knowledge chunks.")


In [ ]:
# 3. Dense Embedding Engine
# For an educational zero-dependency demo, we implement a lightweight semantic hashing
# embedding function that captures semantic character and subword n-gram features.
class EducationalEmbeddingEngine:
    def __init__(self, dim: int = 64):
        self.dim = dim
        
    def embed(self, text: str) -> np.ndarray:
        words = re.findall(r'\w+', text.lower())
        vec = np.zeros(self.dim, dtype=np.float32)
        for w in words:
            # Deterministic hash projection
            h = hash(w) % self.dim
            vec[h] += 1.0
            # Also capture character trigrams for subword robustness
            for i in range(len(w) - 2):
                tri_hash = hash(w[i:i+3]) % self.dim
                vec[tri_hash] += 0.5
        # L2 Normalize
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec

embedder = EducationalEmbeddingEngine(dim=128)

# 4. In-Memory Vector Store
class InMemoryVectorStore:
    def __init__(self):
        self.records: List[Dict[str, Any]] = []
        
    def add(self, chunk_id: str, doc_title: str, text: str, vector: np.ndarray):
        self.records.append({
            "chunk_id": chunk_id,
            "doc_title": doc_title,
            "text": text,
            "vector": vector
        })
        
    def search(self, query_vec: np.ndarray, top_k: int = 2) -> List[Dict[str, Any]]:
        scored = []
        for rec in self.records:
            sim = cosine_similarity(query_vec, rec["vector"])
            scored.append((sim, rec))
            
        # Sort descending by similarity score
        scored.sort(key=lambda x: x[0], reverse=True)
        return [{"score": s, **r} for s, r in scored[:top_k]]

# Populate Vector Store
vector_db = InMemoryVectorStore()
for c in all_chunks:
    vec = embedder.embed(c["text"])
    vector_db.add(c["chunk_id"], c["doc_title"], c["text"], vec)

print(f"Vector Database populated with {len(vector_db.records)} indexed entries.")


In [ ]:
# 5. RAG Retrieval & Prompt Synthesis
def rag_query(user_question: str, top_k: int = 2):
    print(f"\n================ USER QUERY ================")
    print(f"Question: '{user_question}'")
    
    # 1. Embed query
    q_vec = embedder.embed(user_question)
    
    # 2. Retrieve top-k context passages
    retrieved = vector_db.search(q_vec, top_k=top_k)
    
    print(f"\n--- Retrieved Context Passages (Top {top_k}) ---")
    context_str = ""
    for r in retrieved:
        print(f"[{r['doc_title']}] (Similarity Score: {r['score']:.4f})")
        print(f"   Excerpt: {r['text'][:110]}...")
        context_str += f"- Context from [{r['doc_title']}]: {r['text']}\n\n"
        
    # 3. Grounded Prompt Template
    grounded_prompt = f"""System: You are an authoritative AI assistant. Answer the user question strictly using ONLY the factual context provided below. If the answer cannot be determined from the context, state 'Information not found in internal knowledge base.'

Context:
{context_str.strip()}

User Question: {user_question}
Grounded Answer:"""

    print(f"\n--- Synthesized Grounded Prompt for LLM ---")
    print(grounded_prompt)
    return grounded_prompt

# Test our RAG system with two diverse queries
prompt_1 = rag_query("Why do we divide by sqrt(d_k) in attention?", top_k=1)
prompt_2 = rag_query("What optimizer and learning rate schedule should we use?", top_k=1)


---
## 5. Overview of PEFT & LoRA (Low-Rank Adaptation)

When fine-tuning multi-billion parameter models (e.g. Llama-3 8B), updating all 8 billion parameters requires immense GPU VRAM (40GB-80GB+).

**LoRA (Low-Rank Adaptation)** (Hu et al., 2021) freezes the pre-trained model weights $W_0 \in \mathbb{R}^{d \times k}$ and injects trainable **low-rank decomposition matrices** $A$ and $B$:

$$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} (B \times A)$$

Where:
- $W_0 \in \mathbb{R}^{d \times k}$: Frozen pre-trained weight matrix
- $B \in \mathbb{R}^{d \times r}$: Initialized to zeros
- $A \in \mathbb{R}^{r \times k}$: Initialized with random Gaussian noise
- $r \ll \min(d, k)$: The low rank (typically $r \in \{4, 8, 16, 32\}$)
- $\alpha$: Scaling factor

### Why is LoRA so revolutionary?
If $d = 4096$ and $k = 4096$:
- Full matrix $W$: \\$4096 \times 4096 = \mathbf{16,777,216}$ parameters!
- With LoRA rank $r=8$: $(4096 \times 8) + (8 \times 4096) = \mathbf{65,536}$ parameters!
- **Over \\99.6% reduction in trainable parameters** with zero inference latency overhead (since $\Delta W$ can be merged back into $W_0$ during deployment)!


In [ ]:
# Illustrating Parameter Reduction of LoRA
d_in, d_out = 4096, 4096
full_params = d_in * d_out

ranks = [4, 8, 16, 32, 64]
lora_params = [(d_in * r + r * d_out) for r in ranks]
reduction_percentages = [(1.0 - (lp / full_params)) * 100 for lp in lora_params]

print(f"Full Fine-Tuning Parameter Count: {full_params:,}")
print("-" * 55)
for r, lp, red in zip(ranks, lora_params, reduction_percentages):
    print(f"LoRA Rank r={r:2d} | Trainable Params: {lp:,} | Parameter Savings: {red:.2f}%")


---
## 6. Self-Assessment Practice Exercises & Detailed Solutions

### Exercise 1: RAG Chunking Trade-offs
When configuring a RAG pipeline:
1. What happens if your chunk size is too small (e.g. 10 words)?
2. What happens if your chunk size is too large (e.g. 2000 words)?
3. Why is chunk overlap (e.g. 10-20%) critical?

---
### Exercise 2: Hallucination Mitigation
Name 3 concrete engineering techniques used in production RAG systems to detect or prevent hallucinations.

---
### Solutions:
<details>
<summary><b>Click to expand Solution 1</b></summary>

1. **Too Small (10 words)**: Lacks surrounding context and semantic coherence; the embedding cannot capture the complete idea, leading to poor retrieval.
2. **Too Large (2000 words)**: The embedding becomes a diluted average of multiple disparate topics, reducing cosine similarity for specific fine-grained queries, and clutters the LLM's prompt window.
3. **Overlap**: Prevents critical sentences or concepts from being split right at the boundary between two adjacent chunks, preserving semantic continuity!
</details>

<details>
<summary><b>Click to expand Solution 2</b></summary>

1. **Strict System Prompt Instructions**: Constrain the model to answer *only* using provided context, and instruct it to explicitly output a fallback string when the context lacks sufficient evidence.
2. **Source Citation / Attributions**: Require the model to cite the exact chunk ID or sentence index for every claim made.
3. **Self-Correction / Verification Critic**: Run an automated evaluation step (e.g., using an LLM-as-a-Judge or RAGAS metric like Faithfulness) to cross-check if every statement in the generated answer is entailed by the retrieved source chunks.
</details>


---
## 7. Summary & The Modern NLP Frontier
Congratulations! You have journeyed through the entire spectrum of Natural Language Processing:
- **Module 01**: Text Preprocessing, Regex, NLTK, Stemming & Lemmatization
- **Module 02**: Vectorization, Bag of Words, TF-IDF, N-grams, Word2Vec
- **Module 03**: Sequence Modeling, Recurrent Networks, LSTMs, GRUs, BiLSTMs
- **Module 04**: Attention Mechanism, Scaled Dot-Product, Multi-Head Attention, Positional Encoding, Transformers
- **Module 05**: Hugging Face Ecosystem, BERT, RoBERTa, Subword Tokenizers, PyTorch Fine-Tuning
- **Module 06**: Generative AI, Large Language Models, Prompt Engineering, Semantic Search, Vector Stores, RAG, and LoRA!
